# Notebook 04: Linear Models and Regularization

## Why This Matters

Linear models are the workhorses of production ML: they are fast, interpretable, and often surprisingly competitive when features are well-engineered. More importantly, logistic regression and Ridge/Lasso appear in almost every applied ML interview as a baseline and as a lens through which to discuss bias-variance trade-off, regularization, and model interpretability. Understanding why L1 creates sparsity while L2 does not, how to tune the regularization strength, and when to use each variant is foundational knowledge that applies equally to neural networks (weight decay is L2) and to feature selection pipelines.

## 1. Linear Regression from Scratch

Linear regression models the expected target as a linear combination of features:

$$\hat{y} = \mathbf{w}^T \mathbf{x} + b$$

The ordinary least squares (OLS) objective minimizes the mean squared error:

$$\min_{\mathbf{w}, b} \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2$$

The closed-form solution (normal equation): $\mathbf{w}^* = (\mathbf{X}^T \mathbf{X})^{-1} \mathbf{X}^T \mathbf{y}$

The normal equation costs O(p³) to compute (matrix inversion), so gradient descent is preferred when p is large. sklearn uses the Cholesky decomposition for stability.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import fetch_california_housing
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score

housing = fetch_california_housing(as_frame=True)
X = housing.data.values
y = housing.target.values
feature_names = housing.feature_names

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

# Linear regression from scratch using normal equation
class LinearRegressionScratch:
    def fit(self, X, y):
        # Add bias column
        Xb = np.column_stack([np.ones(len(X)), X])
        # Normal equation: (X^T X)^{-1} X^T y
        self.w_ = np.linalg.lstsq(Xb, y, rcond=None)[0]
        return self
    
    def predict(self, X):
        Xb = np.column_stack([np.ones(len(X)), X])
        return Xb @ self.w_

lr_scratch = LinearRegressionScratch().fit(X_train_s, y_train)
y_pred_scratch = lr_scratch.predict(X_test_s)
rmse_scratch = np.sqrt(np.mean((y_test - y_pred_scratch)**2))

from sklearn.linear_model import LinearRegression
lr_sk = LinearRegression().fit(X_train_s, y_train)
rmse_sk = np.sqrt(np.mean((y_test - lr_sk.predict(X_test_s))**2))

print(f"From-scratch RMSE: {rmse_scratch:.4f}")
print(f"sklearn RMSE:      {rmse_sk:.4f}")
print("\nLearned weights:")
for name, w in zip(['bias'] + list(feature_names), lr_scratch.w_):
    print(f"  {name:15s}: {w:+.4f}")

## 2. The Bias-Variance Trade-off

Model error decomposes into three terms:

$$\text{MSE} = \underbrace{\text{Bias}^2}_{\text{underfitting}} + \underbrace{\text{Variance}}_{\text{overfitting}} + \underbrace{\sigma^2_{\text{noise}}}_{\text{irreducible}}$$

- **High bias**: model is too simple; does not capture the true relationship (underfits). Adding features or model complexity reduces bias.
- **High variance**: model is too sensitive to training data; generalizes poorly. Regularization, more data, or simpler model reduces variance.

Regularization is the primary tool for trading off bias and variance in linear models.

In [ ]:
# Visualize overfitting with polynomial regression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline

np.random.seed(42)
n_pts = 30
x_demo = np.linspace(0, 1, n_pts)
y_demo = np.sin(2 * np.pi * x_demo) + 0.2 * np.random.randn(n_pts)
x_line = np.linspace(0, 1, 200)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, deg in zip(axes, [1, 3, 9, 15]):
    pipe = Pipeline([
        ('poly', PolynomialFeatures(deg)),
        ('lr', LinearRegression())
    ])
    pipe.fit(x_demo.reshape(-1, 1), y_demo)
    y_line = pipe.predict(x_line.reshape(-1, 1))
    train_mse = np.mean((y_demo - pipe.predict(x_demo.reshape(-1, 1)))**2)
    ax.scatter(x_demo, y_demo, s=20, color='steelblue', zorder=5)
    ax.plot(x_line, np.sin(2 * np.pi * x_line), 'g--', label='True', alpha=0.6)
    ax.plot(x_line, y_line, 'r-', label=f'deg={deg}')
    ax.set_title(f'Degree {deg}\nTrain MSE={train_mse:.3f}', fontsize=9)
    ax.set_ylim(-2.5, 2.5)
    ax.legend(fontsize=7)

plt.suptitle('Bias-Variance Trade-off via Polynomial Degree', fontsize=12)
plt.tight_layout()
plt.show()

## 3. Ridge Regression (L2 Regularization)

Ridge adds an L2 penalty on the weights:

$$\mathcal{L}_{Ridge} = \frac{1}{n}\sum(y_i - \hat{y}_i)^2 + \alpha \sum_j w_j^2$$

The closed-form solution: $\mathbf{w}^* = (\mathbf{X}^T \mathbf{X} + \alpha \mathbf{I})^{-1} \mathbf{X}^T \mathbf{y}$

Key properties of Ridge:
- **Always invertible**: the $\alpha \mathbf{I}$ term fixes the ill-conditioning problem that arises when features are collinear.
- **Shrinks coefficients toward zero** but almost never to exactly zero — Ridge does not perform feature selection.
- **Distributes shrinkage equally** across correlated features, which is desirable when you want to keep all of them at reduced magnitude.

Geometric intuition: the constraint set is a circle (sphere in high dimensions). The optimal point is where the loss contour first touches the sphere — this almost never happens at an axis (coefficient = 0).

In [ ]:
from sklearn.linear_model import Ridge, RidgeCV

# Ridge coefficient paths
alphas = np.logspace(-3, 4, 100)
coef_paths = []
for a in alphas:
    r = Ridge(alpha=a).fit(X_train_s, y_train)
    coef_paths.append(r.coef_)
coef_paths = np.array(coef_paths)

fig, ax = plt.subplots(figsize=(10, 5))
for i in range(coef_paths.shape[1]):
    ax.plot(np.log10(alphas), coef_paths[:, i], label=feature_names[i])
ax.set_xlabel('log10(alpha)  [higher = stronger regularization]')
ax.set_ylabel('Coefficient value')
ax.set_title('Ridge Regularization Path — Coefficients Shrink but Stay Non-Zero')
ax.legend(fontsize=8, ncol=2, loc='upper right')
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
plt.tight_layout()
plt.show()

In [ ]:
# RidgeCV: efficient leave-one-out cross-validation over alpha grid
ridge_cv = RidgeCV(alphas=np.logspace(-3, 4, 50), cv=5).fit(X_train_s, y_train)
print(f"RidgeCV best alpha: {ridge_cv.alpha_:.4f}")

y_pred_ridge = ridge_cv.predict(X_test_s)
rmse_ridge = np.sqrt(np.mean((y_test - y_pred_ridge)**2))
print(f"Ridge RMSE (best alpha): {rmse_ridge:.4f}")
print(f"OLS RMSE:                {rmse_sk:.4f}")

## 4. Lasso Regression (L1 Regularization)

Lasso adds an L1 penalty:

$$\mathcal{L}_{Lasso} = \frac{1}{n}\sum(y_i - \hat{y}_i)^2 + \alpha \sum_j |w_j|$$

Key properties of Lasso:
- **Produces sparse solutions**: drives many coefficients to exactly zero → automatic feature selection.
- **Selects one from a group of correlated features** and zeros the rest — this is a bug or feature depending on your goals.
- No closed-form solution (the |w| term is not differentiable at zero); solved with coordinate descent.
- Unstable when features are highly correlated: small data changes can swap which correlated feature survives.

**Elastic Net** blends L1 and L2 penalties (`l1_ratio` in sklearn), combining Lasso's sparsity with Ridge's stability on correlated features. It is the recommended default when you want sparse solutions.

In [ ]:
from sklearn.linear_model import Lasso, LassoCV, ElasticNet, ElasticNetCV

# Lasso coefficient path
alphas_lasso = np.logspace(-3, 1, 100)
lasso_paths = []
for a in alphas_lasso:
    m = Lasso(alpha=a, max_iter=5000).fit(X_train_s, y_train)
    lasso_paths.append(m.coef_)
lasso_paths = np.array(lasso_paths)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for i in range(lasso_paths.shape[1]):
    axes[0].plot(np.log10(alphas_lasso), lasso_paths[:, i], label=feature_names[i])
axes[0].axhline(0, color='black', linewidth=0.8, linestyle='--')
axes[0].set_xlabel('log10(alpha)')
axes[0].set_ylabel('Coefficient value')
axes[0].set_title('Lasso Path — Many Coefficients Hit Exactly Zero')
axes[0].legend(fontsize=8)

# Compare number of non-zero features across alpha values
n_nonzero = (np.abs(lasso_paths) > 1e-6).sum(axis=1)
axes[1].plot(np.log10(alphas_lasso), n_nonzero, 'steelblue', marker='o', markersize=3)
axes[1].set_xlabel('log10(alpha)')
axes[1].set_ylabel('Number of non-zero features')
axes[1].set_title('Lasso Sparsity vs. Regularization Strength')

plt.tight_layout()
plt.show()

# Cross-validated Lasso
lasso_cv = LassoCV(cv=5, random_state=42, max_iter=5000).fit(X_train_s, y_train)
print(f"LassoCV best alpha: {lasso_cv.alpha_:.6f}")
print(f"Non-zero features:  {(np.abs(lasso_cv.coef_) > 1e-6).sum()}")
rmse_lasso = np.sqrt(np.mean((y_test - lasso_cv.predict(X_test_s))**2))
print(f"Lasso RMSE:         {rmse_lasso:.4f}")

## 5. Logistic Regression

Logistic regression is a linear model for binary classification. It models the log-odds of the positive class as linear in the features:

$$\log \frac{P(y=1|\mathbf{x})}{1-P(y=1|\mathbf{x})} = \mathbf{w}^T \mathbf{x} + b$$

Equivalently, the probability is a sigmoid applied to the linear combination:

$$P(y=1|\mathbf{x}) = \sigma(\mathbf{w}^T \mathbf{x} + b) = \frac{1}{1 + e^{-(\mathbf{w}^T \mathbf{x} + b)}}$$

Training minimizes the **cross-entropy (log-loss)**:

$$\mathcal{L} = -\frac{1}{n}\sum_i [y_i \log \hat{p}_i + (1-y_i) \log(1-\hat{p}_i)]$$

No closed form — solved iteratively (Newton-CG, LBFGS, liblinear). Regularization (`C = 1/λ`): higher C = less regularization.

In [ ]:
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, classification_report

# Implement sigmoid and log-loss from scratch for intuition
def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

def log_loss_scratch(y_true, y_prob):
    eps = 1e-15
    y_prob = np.clip(y_prob, eps, 1 - eps)
    return -np.mean(y_true * np.log(y_prob) + (1 - y_true) * np.log(1 - y_prob))

# Visualize sigmoid
z = np.linspace(-6, 6, 200)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(z, sigmoid(z), 'steelblue', linewidth=2)
axes[0].axhline(0.5, color='red', linestyle='--', alpha=0.7, label='Decision boundary')
axes[0].axvline(0, color='gray', linestyle=':', alpha=0.7)
axes[0].set_xlabel('z = w·x + b')
axes[0].set_ylabel('σ(z) = P(y=1|x)')
axes[0].set_title('Sigmoid Function')
axes[0].legend()

# Log-loss surface
p = np.linspace(0.001, 0.999, 200)
axes[1].plot(p, -np.log(p),   'steelblue', label='y=1: -log(p)')
axes[1].plot(p, -np.log(1-p), 'coral',     label='y=0: -log(1-p)')
axes[1].set_xlabel('Predicted probability p')
axes[1].set_ylabel('Loss')
axes[1].set_title('Cross-Entropy Loss')
axes[1].legend()
axes[1].set_ylim(0, 5)
plt.tight_layout()
plt.show()

In [ ]:
# Effect of C (regularization) on logistic regression
X_cls, y_cls = make_classification(n_samples=1000, n_features=10, n_informative=5, random_state=42)
X_cls_train, X_cls_test, y_cls_train, y_cls_test = train_test_split(X_cls, y_cls, test_size=0.3, random_state=42)
sc = StandardScaler().fit(X_cls_train)
X_cls_train_s = sc.transform(X_cls_train)
X_cls_test_s  = sc.transform(X_cls_test)

C_vals = np.logspace(-3, 3, 20)
train_aucs, test_aucs, n_nonzero = [], [], []

for C in C_vals:
    m = LogisticRegression(C=C, penalty='l1', solver='liblinear', max_iter=500)
    m.fit(X_cls_train_s, y_cls_train)
    train_aucs.append(roc_auc_score(y_cls_train, m.predict_proba(X_cls_train_s)[:, 1]))
    test_aucs.append(roc_auc_score(y_cls_test, m.predict_proba(X_cls_test_s)[:, 1]))
    n_nonzero.append((np.abs(m.coef_[0]) > 1e-6).sum())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].semilogx(C_vals, train_aucs, 'steelblue', label='Train AUC', marker='o', markersize=4)
axes[0].semilogx(C_vals, test_aucs,  'coral', label='Test AUC', marker='s', markersize=4)
axes[0].set_xlabel('C (inverse regularization)')
axes[0].set_ylabel('AUC')
axes[0].set_title('L1 Logistic: AUC vs C')
axes[0].legend()

axes[1].semilogx(C_vals, n_nonzero, 'purple', marker='D', markersize=4)
axes[1].set_xlabel('C')
axes[1].set_ylabel('Non-zero features')
axes[1].set_title('L1 Logistic: Sparsity vs C')

plt.tight_layout()
plt.show()

## 6. Ridge vs Lasso vs Elastic Net — Geometric Intuition

The key difference between L1 and L2 is the shape of the constraint region. In the 2D case:
- **L2 (Ridge)**: constraint is a circle — smooth, no corners. The optimal solution (intersection of loss contour and constraint set) almost never sits on an axis.
- **L1 (Lasso)**: constraint is a diamond — has corners on the axes. The optimal solution often lands on a corner where w_j = 0.

This geometric intuition directly explains why Lasso produces sparsity and Ridge does not, and it is a standard interview question.

In [ ]:
# Visualize L1 vs L2 constraint geometry
fig, axes = plt.subplots(1, 2, figsize=(10, 5))

theta = np.linspace(0, 2 * np.pi, 300)

for ax, title, constraint_x, constraint_y, color in [
    (axes[0], 'Ridge (L2): Circle Constraint', np.cos(theta), np.sin(theta), 'steelblue'),
    (axes[1], 'Lasso (L1): Diamond Constraint',
     np.cos(theta) / (np.abs(np.cos(theta)) + np.abs(np.sin(theta))),
     np.sin(theta) / (np.abs(np.cos(theta)) + np.abs(np.sin(theta))), 'coral'),
]:
    # Constraint region
    ax.fill(constraint_x, constraint_y, alpha=0.3, color=color, label='Constraint region')
    ax.plot(constraint_x, constraint_y, color=color, linewidth=2)
    
    # Elliptical loss contours (centered off-axis to show the intersection)
    for level in [0.5, 1.0, 1.5, 2.0]:
        loss_theta = np.linspace(0, 2*np.pi, 300)
        lx = 1.2 + level * 0.8 * np.cos(loss_theta)
        ly = 0.8 + level * 0.5 * np.sin(loss_theta)
        ax.plot(lx, ly, 'gray', alpha=0.4, linewidth=0.8)
    
    ax.set_xlim(-2.5, 3)
    ax.set_ylim(-2, 2)
    ax.axhline(0, color='black', linewidth=0.5)
    ax.axvline(0, color='black', linewidth=0.5)
    ax.set_xlabel('w₁')
    ax.set_ylabel('w₂')
    ax.set_title(title, fontsize=10)
    ax.set_aspect('equal')

plt.suptitle('L1 corners → sparse solutions; L2 circle → no sparsity', fontsize=11)
plt.tight_layout()
plt.show()

## Summary

| Model | Penalty | Sparsity | Handles Correlated Features | Closed Form | Use When |
|-------|---------|----------|----------------------------|-------------|----------|
| OLS Linear Regression | None | No | Poor (blows up) | Yes | Baseline; well-conditioned features |
| Ridge (L2) | α Σ w² | No | Yes (shrinks together) | Yes (+αI) | Many small effects; correlated features |
| Lasso (L1) | α Σ \|w\| | Yes | Poor (selects one) | No | Sparse feature selection needed |
| Elastic Net | α[ρΣ\|w\| + (1-ρ)Σw²] | Yes | Yes | No | Best of both: sparsity + stability |
| Logistic Regression | Optional L1/L2 | With L1 | Depends | No | Binary/multiclass classification |